# DP2 Batch Injection Demo

This notebook runs ten rounds of 100 synthetic clusters on one DP2 coadd, detects them with MCI, matches detections to injected sources, and plots per-round and combined completeness. Run from top to bottom in the CANFAR Rubin kernel.

## 1. Imports
Use your Rubin kernel with INJECT installed or importable from the checkout.

In [ ]:
from pathlib import Path
import sys
repo = Path('/arc/home/snair25/INJECT')
if repo.is_dir():
    sys.path.insert(0, str(repo))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler
from inject import ClusterConfig, ClusterRetrieval, InjectionConfig, InjectionPipeline
from inject.detection import run_cluster_detection
from inject.plotting import plot_completeness_1d, plot_completeness_2d

## 2. Settings
Edit the DP2 data ID if needed. This example injects 100 sources per round for ten rounds.

In [ ]:
REPO = 'https://ws-uv.canfar.net/lsst/api/butler/repo/dp2/butler.yaml'
COLLECTION = 'dp2'
DATA_ID = {'skymap': 'lsst_cells_v2', 'tract': 7508, 'patch': 0, 'band': 'r'}
N_PER_ROUND = 100
N_ROUNDS = 10
MATCH_RADIUS = 5.0
OUT = Path('plots/dp2_batch_injection_demo')
OUT.mkdir(parents=True, exist_ok=True)
config = InjectionConfig(
    band=DATA_ID['band'], n_clusters=N_PER_ROUND, cutout_size=800,
    edge_buffer=60, seed=42, use_actual_psf=True, add_noise=False,
    pixel_scale=0.2, zero_point=31.4, output_dir=str(OUT),
    cluster_config=ClusterConfig(
        profile_type='king', mag_min=20.5, mag_max=24.5,
        r_half_min=2, r_half_max=6, concentration_min=5, concentration_max=20,
    ),
)

## 3. Load DP2
Load one coadd; each round injects into a fresh copy of this image.

In [ ]:
pipe = InjectionPipeline(config)
pipe.load_data(butler=Butler(REPO, collections=COLLECTION),
               data_id=DATA_ID, dataset_type='deep_coadd')
print('Image shape:', pipe.image.shape)

## 4. MCI detector
The detector returns pixel-coordinate catalogs for the pipeline to match.

In [ ]:
def detect(image):
    return run_cluster_detection(
        image, psf_fwhm=config.psf_fwhm_fallback,
        pixel_scale=config.pixel_scale, threshold_sigma=3, snr_min=3,
        mci_max=0.85, r_half_min=1, ellipticity_max=0.6,
        box_size=64, use_multiscale=True, use_mci=True, verbose=False,
    )

## 5. Run the batch
Run one injection and detection per round. Each round is matched independently; pooled completeness combines those already matched results.

In [ ]:
bbox_x, bbox_y = pipe.bboxes[config.band]
baseline_detections = detect(pipe.image)
runs = pipe.run_batch(
    n_iterations=N_ROUNDS, n_per_iter=N_PER_ROUND,
    psf_obj=pipe.psf_objs[config.band], bbox_x_min=bbox_x, bbox_y_min=bbox_y,
    mask_array=pipe.masks[config.band], mask_plane_dict=pipe.mask_plane_dicts[config.band],
    detector_fn=detect, checkpoint_dir=str(OUT / 'checkpoints'),
    n_workers=1, use_psf_cache=False, store_images=True,
    band=config.band, verbose=False,
)
print('Rounds:', len(runs))
print('Injected per round:', [len(run['injection_info']) for run in runs])

## 6. Plot each round and combined completeness
`make_plots()` saves the per-round images and position maps. Matches are one-to-one within `MATCH_RADIUS` pixels and kept separate by round. Pool those matched truth rows for combined 1D and 2D completeness.

In [ ]:
matched_rows = []
for run in runs:
    retrieval = ClusterRetrieval(run['injection_info'], run['detections'])
    retrieval.match_detections(
        match_radius=MATCH_RADIUS, baseline_detections=baseline_detections,
    )
    matched_rows.extend(retrieval.injection_info)

    pipe.injected_image = run['injected_image']
    pipe.injection_info = retrieval._matched
    pipe.detection_catalog = run['detections']
    pipe.retrieval = retrieval
    pipe.make_plots(
        output_dir=str(OUT / f"round_{run['iteration'] + 1:02d}"),
        plots=['before_after', 'position_map'], show=False, save=True, n_stamps=3,
    )

pooled = ClusterRetrieval.from_matched(matched_rows)
fig, _ = plot_completeness_1d(pooled, config)
fig.savefig(OUT / 'completeness_1d_all_rounds.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)
fig, _ = plot_completeness_2d(pooled, config)
fig.savefig(OUT / 'completeness_2d_all_rounds.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)
pd.DataFrame([pooled.get_summary_statistics()])

The batch API uses the lightweight array injector, so it does not update the Rubin variance plane. Each round's detections are matched only against that round's injections.